In [9]:
import os
import re
from nilearn import datasets

target_dir = "/mnt/movement/users/jaizor/xtra/data/fmri"

# 1. Fetch URLs
urls_path, all_urls = datasets.fetch_ds000030_urls(data_dir=target_dir, verbose=0)

# 2. STRICT FILTERING
# Must contain 'sub-', 'func', 'task-rest', AND end with .nii.gz, .json, or _events.tsv
# MUST NOT contain 'derivatives'
fmri_urls = []
for url in all_urls:
    if 'derivatives' in url:
        continue # Skip all preprocessed data
    if 'sub-' in url and 'func' in url and 'task-rest' in url:
        if url.endswith(('.nii.gz', '.json', '_events.tsv')):
            fmri_urls.append(url)

print(f"✅ Selected {len(fmri_urls)} files (Raw Resting State ONLY).")
print("Sample URLs:")
for u in fmri_urls[:3]: print(f"  - {u}")

# 3. Download ONLY these specific files
# We pass the explicit list. Nilearn should only grab these.
data_dir, downloaded_files = datasets.fetch_openneuro_dataset(
    urls=fmri_urls, 
    dataset_version='ds000030_R1.0.4',
    data_dir=target_dir,
    verbose=1
)

✅ Selected 799 files (Raw Resting State ONLY).
Sample URLs:
  - https://s3.amazonaws.com/openneuro/ds000030/ds000030_R1.0.4/uncompressed/sub-10159/func/sub-10159_task-rest_bold.json
  - https://s3.amazonaws.com/openneuro/ds000030/ds000030_R1.0.4/uncompressed/sub-10159/func/sub-10159_task-rest_bold.nii.gz
  - https://s3.amazonaws.com/openneuro/ds000030/ds000030_R1.0.4/uncompressed/sub-10171/func/sub-10171_task-rest_bold.json


[fetch_openneuro_dataset] Dataset found in 
/mnt/movement/users/jaizor/xtra/data/fmri/ds000030/ds000030_R1.0.4/uncompressed

In [10]:
import pandas as pd
from pathlib import Path

# Paths
DATA_ROOT = Path("/mnt/movement/users/jaizor/xtra/data/fmri/ds000030/ds000030_R1.0.4/uncompressed")
TSV_PATH = DATA_ROOT / "participants.tsv"
OUTPUT_CSV = Path("/mnt/movement/users/jaizor/xtra/data/fmri/resting_state_master_manifest.csv")

# 1. Load Phenotypes
print("🦖 Loading phenotypes...")
df_pheno = pd.read_csv(TSV_PATH, sep="\t")

# Clean participant_id in TSV to ensure it's just the number (string)
# Sometimes TSV has 'sub-XXXXX', sometimes just 'XXXXX'. Let's standardize to just numbers.
df_pheno['participant_id_clean'] = df_pheno['participant_id'].astype(str).str.replace('sub-', '')

print(f"✅ Loaded {len(df_pheno)} subjects.")
print(f"   Diagnosis distribution:\n{df_pheno['diagnosis'].value_counts()}")

# 2. Find Resting State Files
print("\n🔍 Scanning for resting-state NIfTIs...")
nifti_files = list(DATA_ROOT.glob("sub-*/func/*_task-rest_bold.nii.gz"))
print(f"   Found {len(nifti_files)} files.")

# 3. Build Master DataFrame
records = []
missing_count = 0

for nifti_path in nifti_files:
    # Extract ID from path: e.g., ".../sub-60028/func/..." -> "60028"
    subject_id_full = nifti_path.parent.parent.name # "sub-60028"
    subject_id_clean = subject_id_full.replace('sub-', '') # "60028"
    
    # Merge with Phenotypes
    row = df_pheno[df_pheno['participant_id_clean'] == subject_id_clean]
    
    if row.empty:
        print(f"⚠️ Warning: No phenotype found for {subject_id_full}")
        missing_count += 1
        continue
    
    diagnosis = row['diagnosis'].values[0]
    
    # Optional: Verify the 'rest' flag is 1 (sanity check)
    has_rest_scan = row['rest'].values[0] if 'rest' in row.columns else True
    
    records.append({
        'subject_id': subject_id_full,
        'file_path': str(nifti_path.absolute()),
        'diagnosis': diagnosis,
        'age': row['age'].values[0],
        'gender': row['gender'].values[0],
        'has_rest_flag': has_rest_scan
    })

df_master = pd.DataFrame(records)

# 4. Save
df_master.to_csv(OUTPUT_CSV, index=False)
print(f"\n🎉 Success! Manifest saved to: {OUTPUT_CSV}")
print(f"   Total samples: {len(df_master)}")
print(f"   Missing phenotypes: {missing_count}")

if len(df_master) > 0:
    print("\n📊 Final Label Distribution:")
    print(df_master['diagnosis'].value_counts())
    print("\nFirst 5 rows:")
    print(df_master.head())
else:
    print("\n❌ ERROR: Manifest is empty! Check ID matching logic.")

🦖 Loading phenotypes...
✅ Loaded 272 subjects.
   Diagnosis distribution:
diagnosis
CONTROL    130
SCHZ        50
BIPOLAR     49
ADHD        43
Name: count, dtype: int64

🔍 Scanning for resting-state NIfTIs...
   Found 268 files.

🎉 Success! Manifest saved to: /mnt/movement/users/jaizor/xtra/data/fmri/resting_state_master_manifest.csv
   Total samples: 268
   Missing phenotypes: 0

📊 Final Label Distribution:
diagnosis
CONTROL    127
SCHZ        50
BIPOLAR     49
ADHD        42
Name: count, dtype: int64

First 5 rows:
  subject_id                                          file_path diagnosis  \
0  sub-60028  /mnt/movement/users/jaizor/xtra/data/fmri/ds00...   BIPOLAR   
1  sub-50027  /mnt/movement/users/jaizor/xtra/data/fmri/ds00...      SCHZ   
2  sub-10530  /mnt/movement/users/jaizor/xtra/data/fmri/ds00...   CONTROL   
3  sub-50034  /mnt/movement/users/jaizor/xtra/data/fmri/ds00...      SCHZ   
4  sub-50005  /mnt/movement/users/jaizor/xtra/data/fmri/ds00...      SCHZ   

   age gender

In [ ]:
import pandas as pd

# --- Load your existing manifest ---
input_csv = "/mnt/movement/users/jaizor/xtra/data/fmri/resting_state_master_manifest.csv"
df = pd.read_csv(input_csv)

print("🦖 Loading manifest...")
print(f"   Total samples: {len(df)}")

# --- Create binary diagnosis columns ---
df['control'] = (df['diagnosis'] == 'CONTROL').astype(int)
df['schz']    = (df['diagnosis'] == 'SCHZ').astype(int)
df['bipolar'] = (df['diagnosis'] == 'BIPOLAR').astype(int)
df['adhd']    = (df['diagnosis'] == 'ADHD').astype(int)

# --- Process eid, Age, and Sex ---
# Extract eid by removing 'sub-' prefix
df['eid'] = df['subject_id'].str.replace('sub-', '')

# Rename 'age' to 'Age'
df['Age'] = df['age']

# Encode 'gender' as Sex: 1 = M, 0 = F
df['Sex'] = df['gender'].map({'M': 1, 'F': 0})

# --- Select final columns in the requested order ---
final_df = df[['eid', 'Age', 'Sex', 'control', 'schz', 'bipolar', 'adhd']].copy()

# --- Save the clean CSV ---
output_csv = "/mnt/movement/users/jaizor/xtra/data/fmri//pheno_ucla.csv"
final_df.to_csv(output_csv, index=False)

print(f"\n✅ Clean phenotype CSV saved to: {output_csv}")
print("\n📊 First 5 rows:")
print(final_df.head())

# Final validation
print("\n📈 Label Counts:")
print(f"  CONTROL : {final_df['control'].sum()}")
print(f"  SCHZ    : {final_df['schz'].sum()}")
print(f"  BIPOLAR : {final_df['bipolar'].sum()}")
print(f"  ADHD    : {final_df['adhd'].sum()}")

print(f"\n🧬 Sex encoding: 1 = M, 0 = F")

🦖 Loading manifest...
   Total samples: 268

✅ Clean phenotype CSV saved to: /mnt/movement/users/jaizor/xtra/data/fmri/pheno_ucla.csv

📊 First 5 rows:
     eid  Age  Sex  control  schz  bipolar  adhd
0  60028   36    0        0     0        1     0
1  50027   30    0        0     1        0     0
2  10530   23    1        1     0        0     0
3  50034   43    1        0     1        0     0
4  50005   40    1        0     1        0     0

📈 Label Counts:
  CONTROL : 127
  SCHZ    : 50
  BIPOLAR : 49
  ADHD    : 42

🧬 Sex encoding: 1 = M, 0 = F
